In [ ]:
# Barrido de parámetros — priorización de genoma completo

Validación **leave-one-species-out** sobre las 16 especies del modelo: para cada
una se arma la semilla de druggables *sin* su evidencia, se propaga sobre la capa
de afiliaciones y se mide cuán arriba del ranking quedan sus blancos conocidos.

La grilla es `alpha × beta × lambda × gamma` = **120 combinaciones válidas** por
especie (cuando `lambda` está fijo el modelo no usa `gamma`, así que esas
repeticiones se descartan). La métrica es la AUC parcial en `FPR ≤ 0.10`,
normalizada y corregida por McClish: azar → 0.5, perfecto → 1.0.

Salidas en `gon4/genome_prioritization_out/`: `01_<especie>.csv` (una fila por
combinación) y `01_meta.json`.

**Control** (README §1.3): los óptimos tienen que coincidir con los de
`gon3/resultados/genoma_completo_v5/`, que ya usa la métrica corregida. La
comparación está en la celda 7.

## Imports

In [ ]:
import sys, os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(_v, "1")          # antes de numpy: un hilo por proceso

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns

from pathlib import Path
# raiz del repositorio: se busca hacia arriba la carpeta que tiene DB/,
# asi el notebook corre desde donde sea que se haya clonado
RAIZ = Path.cwd()
while not (RAIZ / "DB").is_dir() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ / "comun"))
sys.path.insert(0, str(RAIZ / "genome_prioritization"))
%load_ext autoreload
%autoreload 2
import tdr, nucleo as nf
import funciones_genome as fg              # el .py de esta carpeta

SALIDAS = tdr.out("genome_prioritization")
FIGURAS = SALIDAS / "figuras"
NB      = "01"                             # numero de este notebook: prefija todo lo que genere
n_core  = 20

plt = tdr.estilo()
SALIDAS

## Datos

In [ ]:
# Sin cluster_consistent: el barrido usa todos los positivos, como full_genome_v5.
datos = tdr.cargar_db(anotaciones=True, cluster_consistent=False)
datos.st.shape, datos.posdt.shape, datos.sta.shape

## Acondicionamiento

In [ ]:
spoi   = datos.especies_con_druggables(minimo=10)      # las 16 del modelo
params = fg.PARAMS
combos = fg.grilla_valida(params)

entrada = fg.resumen_especies(datos, spoi)
print(f"{len(spoi)} especies x {len(combos)} combinaciones = {len(spoi)*len(combos)} corridas de nds()")
entrada[["sp_id", "nombre", "grupo", "N", "N_druggable"]]

In [ ]:
# Vista de los datos de entrada: sobre qué se va a correr el barrido.
fig, ax = plt.subplots(figsize=(7, 3.5), tight_layout=True)
e = entrada.sort_values("N_druggable")
ax.barh(range(len(e)), e["N_druggable"], color=tdr.S1)
ax.set_yticks(range(len(e)))
ax.set_yticklabels([tdr.NOMBRE_CORTO.get(s, s) for s in e["sp_id"]], fontsize=8)
ax.set_xscale("log")
ax.set_xlabel("blancos druggables (positivos a recuperar)")
ax.set_title("Positivos por especie")
tdr.guardar(fig, f"{NB}_f01_positivos_por_especie", FIGURAS)

## Corrida

In [ ]:
# celda de corrida: el ciclo se lee aca
import time
for sp in spoi:
    t0  = time.time()
    ctx = fg.preparar_especie(datos, sp)                          # semilla LOSO + cat_rs
    ctx["rs"] = fg.relevance_por_alpha(ctx, params["alpha"])      # RS por alpha, una vez
    fg.fijar_contexto(datos.sta, ctx)                             # visible por fork
    filas = fg.paralelizar(fg.evaluar_combinacion, combos, n_core=n_core, desc=sp)
    fg.guardar_barrido(filas, SALIDAS, NB, sp, params)            # -> 01_<sp>.csv
    print(f"  {sp} en {time.time() - t0:.0f} s", flush=True)

fg.escribir_meta(SALIDAS, NB, notebook="01_barrido_parametros.ipynb",
                 params=params, n_core=n_core, especies=spoi,
                 cluster_consistent=False, metrica="AUC01 = McClish(pAUC(FPR<=0.10))")

# Resultados

In [ ]:
# lee de gon4/, no recalcula: esta celda no depende de la de corrida
barrido = fg.cargar_barrido(SALIDAS, NB)
optimos = fg.optimos_por_especie(barrido)
print(tdr.leer_meta(SALIDAS, NB)["fecha"])
optimos[["especie", "AUC01", "AUC", "alpha", "beta", "lambda_", "gamma", "N_targets"]]

In [ ]:
# Control (README §1.3): los optimos nuevos contra genoma_completo_v5
control = fg.comparar_con_control(optimos)
print(f"coinciden los parametros optimos en {control['coincide_params'].sum()} de {len(control)} especies")
print(f"|delta AUC01| maximo: {control['delta'].abs().max():.2e}")
control

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 4.5), tight_layout=True)
ax.scatter(control["auc01_v5"], control["auc01_v4"], color=tdr.S1, zorder=3)
lims = [0.5, 1.0]
ax.plot(lims, lims, color=tdr.MUTED, ls="--", lw=1)
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel("AUC01 — genoma_completo_v5 (control)")
ax.set_ylabel("AUC01 — v4")
ax.set_title("Los números nuevos reproducen el control")
tdr.guardar(fig, f"{NB}_f02_control_v5", FIGURAS)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5), tight_layout=True)
o = optimos.sort_values("AUC01")
ax.barh(range(len(o)), o["AUC01"], color=tdr.S1)
ax.axvline(0.5, color=tdr.MUTED, ls="--", lw=1)
ax.set_yticks(range(len(o)))
ax.set_yticklabels([tdr.NOMBRE_CORTO.get(s, s) for s in o["especie"]], fontsize=8)
ax.set_xlim(0.5, 1.0)
ax.set_xlabel("AUC01 en el óptimo  (0.5 = azar)")
ax.set_title("Priorización por especie, LOSO")
tdr.guardar(fig, f"{NB}_f03_auc01_por_especie", FIGURAS)